# Session 3: Can machine learning beat MIDAS for Germany?

This notebook keeps the **same country (`DEU`)**, **same target**, and **same rolling out-of-sample design** as the previous notebook. The only thing that changes is the model class.

That makes the comparison fair.

## Learning goals

By the end of the notebook you should be able to:

1. build a richer feature matrix from mixed-frequency macro data,
2. compare MIDAS with tree-based ML regressors,
3. inspect feature importance and SHAP values,
4. understand why ML may help in some settings and fail in others.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression

from MIDAS import (
    PALETTE,
    feature_importance_frame,
    make_lags,
    rmse,
    mae,
    rolling_beta_midas_forecast,
    shap_importance_frame,
    stack_midas_features,
    use_aer_style,
)

warnings.filterwarnings('ignore')
use_aer_style()

COUNTRY = 'DEU'
TARGET = 'gdp_yoy'
N_LAGS = 12
END_MONTH = 3
EVAL_START = '2005-01-01'


## 1. Load the same Germany data

We keep the exact same target as in Workshop 1: quarterly GDP YoY growth.

The difference is that now we allow ML models to see a **larger set of monthly lagged features**:

- industrial production,
- inflation,
- unemployment,
- short-term interest rates,
- plus a couple of lagged GDP terms.

That makes the feature space richer, but it also raises the risk of overfitting because the quarterly sample is still small.


In [ ]:
q = pd.read_csv('data/deu_quarterly.csv', parse_dates=['date'])
m = pd.read_csv('data/deu_monthly.csv', parse_dates=['date'])

q = q[q['country'] == COUNTRY].sort_values('date').reset_index(drop=True)
m = m[m['country'] == COUNTRY].sort_values('date').reset_index(drop=True)

q[['date', 'gdp_yoy', 'ip_yoy', 'cpi_yoy', 'unemp', 'rate3m']].dropna().head(8)


## 2. Engineer a richer feature matrix

For MIDAS we used one monthly series and a smooth weight curve.

For ML we will hand the model a larger tabular feature matrix built from:

- 12 monthly lags of `ip_yoy`, `cpi_yoy`, `unemp`, and `rate3m`,
- 2 quarterly lags of GDP growth.

This is exactly the kind of setup where tree models *might* help:

- nonlinearities,
- threshold effects,
- interactions across predictors.

But it is also the kind of setup where they can fail if the sample is too short.


In [ ]:
base = q[['date', 'country', 'gdp_yoy']].copy()
base = make_lags(base, ['gdp_yoy'], [1, 2], by=None)

monthly_macro = (
    m[['date', 'ip_yoy', 'cpi_yoy', 'unemp', 'rate3m']]
    .dropna(subset=['ip_yoy', 'cpi_yoy', 'unemp', 'rate3m'])
    .set_index('date')
    .sort_index()
)

stacked = stack_midas_features(
    monthly_macro,
    pd.DatetimeIndex(base['date']),
    ['ip_yoy', 'cpi_yoy', 'unemp', 'rate3m'],
    n_lags=N_LAGS,
    end_month=END_MONTH,
)

ml_frame = pd.concat([base.reset_index(drop=True), stacked.reset_index(drop=True)], axis=1)
ml_features = [c for c in ml_frame.columns if c.endswith(tuple([f'L{i}' for i in range(N_LAGS)]))] + ['gdp_yoy_l1', 'gdp_yoy_l2']

print('Quarterly observations available:', len(ml_frame))
print('Number of ML features:', len(ml_features))
ml_frame[['date', 'gdp_yoy', 'gdp_yoy_l1', 'gdp_yoy_l2'] + ml_features[:6]].head()


### A useful warning sign

If you have roughly 70–80 evaluation observations and 50+ features, you should already be suspicious.

Machine learning is not magic. A very flexible model can still do badly if:

- the sample is too short,
- the signal is mostly linear,
- the important dynamics are already captured by a parsimonious econometric model.

## 3. Define the competitor set

We will compare:

1. **AR(1)** — a minimal benchmark.
2. **Bridge (macro averages)** — quarterly averages + GDP persistence.
3. **ADL-MIDAS (macro + AR)** — the econometric benchmark.
4. **Random Forest** — flexible nonlinear regression.
5. **Histogram Gradient Boosting** — a modern boosted-tree baseline from scikit-learn.

Notice the design choice here: the ML models get a **larger raw feature matrix**, but MIDAS gets a **more structured restriction**.


In [ ]:
def rolling_ols_forecast(frame, target, features, eval_start='2005-01-01', min_train=24):
    use = frame[['date', target] + features].dropna().copy().sort_values('date')
    rows = []
    for dt in sorted(use.loc[use['date'] >= pd.Timestamp(eval_start), 'date'].unique()):
        train = use[use['date'] < dt]
        test = use[use['date'] == dt]
        if len(train) < min_train or test.empty:
            continue
        model = LinearRegression().fit(train[features], train[target])
        out = test[['date', target]].copy()
        out['y_hat'] = model.predict(test[features])
        rows.append(out)
    return pd.concat(rows, ignore_index=True).rename(columns={target: 'y_true'})


def rolling_ml_forecast(frame, target, features, model_factory, eval_start='2005-01-01', min_train=24):
    use = frame[['date', target] + features].dropna().copy().sort_values('date')
    rows = []
    for dt in sorted(use.loc[use['date'] >= pd.Timestamp(eval_start), 'date'].unique()):
        train = use[use['date'] < dt]
        test = use[use['date'] == dt]
        if len(train) < min_train or test.empty:
            continue
        model = model_factory()
        model.fit(train[features], train[target])
        out = test[['date', target]].copy()
        out['y_hat'] = model.predict(test[features])
        rows.append(out)
    return pd.concat(rows, ignore_index=True).rename(columns={target: 'y_true'})


def common_sample(forecasts):
    merged = None
    for name, df in forecasts.items():
        sub = df[['date', 'y_true', 'y_hat']].rename(columns={'y_hat': name})
        if merged is None:
            merged = sub.copy()
        else:
            merged = merged.merge(sub[['date', name]], on='date', how='inner')
    return merged.rename(columns={'y_true': 'actual'})


In [ ]:
# Quarterly baselines
quarterly_baselines = q[['date', 'gdp_yoy', 'ip_yoy', 'cpi_yoy', 'unemp', 'rate3m']].copy()
quarterly_baselines = make_lags(quarterly_baselines, ['gdp_yoy'], [1], by=None)
for var in ['ip_yoy', 'cpi_yoy', 'unemp', 'rate3m']:
    quarterly_baselines[f'{var}_qavg_l0'] = q[var].values

# MIDAS frame with 4 monthly predictors + one GDP lag
midas_macro_frame = base.copy()
midas_macro_frame = make_lags(midas_macro_frame, ['gdp_yoy'], [1], by=None)
stacked_macro = stack_midas_features(
    monthly_macro,
    pd.DatetimeIndex(midas_macro_frame['date']),
    ['ip_yoy', 'cpi_yoy', 'unemp', 'rate3m'],
    n_lags=N_LAGS,
    end_month=END_MONTH,
)
midas_macro_frame = pd.concat([midas_macro_frame.reset_index(drop=True), stacked_macro.reset_index(drop=True)], axis=1)

fc_ar1 = rolling_ols_forecast(quarterly_baselines, 'gdp_yoy', ['gdp_yoy_l1'], eval_start=EVAL_START)
fc_bridge = rolling_ols_forecast(
    quarterly_baselines,
    'gdp_yoy',
    ['gdp_yoy_l1', 'ip_yoy_qavg_l0', 'cpi_yoy_qavg_l0', 'unemp_qavg_l0', 'rate3m_qavg_l0'],
    eval_start=EVAL_START,
)
fc_midas = rolling_beta_midas_forecast(
    midas_macro_frame,
    'gdp_yoy',
    ['ip_yoy', 'cpi_yoy', 'unemp', 'rate3m'],
    n_lags=N_LAGS,
    low_freq_features=['gdp_yoy_l1'],
    eval_start=EVAL_START,
    min_train=24,
)
fc_rf = rolling_ml_forecast(
    ml_frame,
    'gdp_yoy',
    ml_features,
    lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=0, n_jobs=1),
    eval_start=EVAL_START,
)
fc_hgb = rolling_ml_forecast(
    ml_frame,
    'gdp_yoy',
    ml_features,
    lambda: HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, random_state=0),
    eval_start=EVAL_START,
)

forecast_dict = {
    'AR(1)': fc_ar1,
    'Bridge (macro avgs)': fc_bridge,
    'ADL-MIDAS (macro + AR)': fc_midas,
    'Random Forest': fc_rf,
    'HistGradientBoosting': fc_hgb,
}


In [ ]:
aligned = common_sample(forecast_dict)
metrics = []
for name in forecast_dict:
    metrics.append({'model': name, 'RMSE': rmse(aligned['actual'], aligned[name]), 'MAE': mae(aligned['actual'], aligned[name])})
metrics = pd.DataFrame(metrics).sort_values('RMSE').reset_index(drop=True)
metrics


## 4. Scoreboard: did ML earn its flexibility?

This is the crucial teaching moment.

If MIDAS stays ahead, that does **not** mean ML is useless. It means that in this particular sample:

- the dataset is short,
- the macro signal is structured,
- and a parsimonious econometric restriction may be more valuable than raw flexibility.


In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 3.8))
color_map = {
    'AR(1)': PALETTE['pastgray'],
    'Bridge (macro avgs)': PALETTE['myblue'],
    'ADL-MIDAS (macro + AR)': PALETTE['mygreen'],
    'Random Forest': PALETTE['myred'],
    'HistGradientBoosting': '#9b5de5',
}
ax.bar(metrics['model'], metrics['RMSE'], color=[color_map[m] for m in metrics['model']])
ax.set_ylabel('RMSE (pp)')
ax.set_title('Germany: MIDAS vs. ML on the same rolling sample')
ax.grid(True, axis='y')
plt.xticks(rotation=12, ha='right')
fig.tight_layout()
plt.show()


In [ ]:
best_benchmark = metrics.loc[metrics['model'] == 'ADL-MIDAS (macro + AR)', 'RMSE'].iloc[0]
challenge_text = f'Beat ADL-MIDAS RMSE = {best_benchmark:.3f} without leaking future information.'
challenge_text


## 5. Crisis windows: where do the models differ most?

Average RMSE can hide a lot. The next chart focuses on the periods students usually care about most:

- the Global Financial Crisis,
- COVID.

This is also where nonlinear models are often *supposed* to shine.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.8, 6.0), sharex=False)
plot_specs = [
    ('2006-01-01', '2012-12-31', 'Global Financial Crisis window'),
    ('2018-01-01', '2022-12-31', 'COVID window'),
]
colors = {
    'actual': PALETTE['pastgray'],
    'ADL-MIDAS (macro + AR)': PALETTE['mygreen'],
    'Random Forest': PALETTE['myred'],
    'HistGradientBoosting': '#9b5de5',
}
for ax, (start, end, title) in zip(axes, plot_specs):
    sub = aligned[(aligned['date'] >= pd.Timestamp(start)) & (aligned['date'] <= pd.Timestamp(end))].copy()
    ax.plot(sub['date'], sub['actual'], color=colors['actual'], lw=1.8, label='Actual GDP YoY')
    for name in ['ADL-MIDAS (macro + AR)', 'Random Forest', 'HistGradientBoosting']:
        ax.plot(sub['date'], sub[name], lw=1.4, label=name, color=colors[name])
    ax.axhline(0, color='#666666', lw=0.7)
    ax.set_title(title)
    ax.set_ylabel('YoY growth (%)')
    ax.grid(True)
axes[0].legend(ncol=2, frameon=False, loc='upper right')
axes[-1].set_xlabel('Quarter')
fig.tight_layout()
plt.show()


## 6. Interpretability: what is the Random Forest paying attention to?

For interpretation we fit a Random Forest on the full available Germany sample with complete features.

Two different questions are useful here:

- **feature importance**: which variables the forest uses most often,
- **SHAP values**: which variables have the largest average marginal contribution.

In small macro samples, these tools are often more informative than the headline accuracy table.


In [ ]:
train_full = ml_frame[['gdp_yoy'] + ml_features].dropna().copy()
X_full = train_full[ml_features]
y_full = train_full['gdp_yoy']

rf_full = RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=0, n_jobs=1)
rf_full.fit(X_full, y_full)

rf_imp, imp_method = feature_importance_frame(rf_full, X_full.values, y_full.values, ml_features, top_n=12)
try:
    rf_shap = shap_importance_frame(rf_full, X_full.values[:min(120, len(X_full))], ml_features, top_n=12)
    shap_method = 'SHAP'
except Exception as exc:
    print(f'SHAP is unavailable in this environment: {exc}')
    rf_shap = rf_imp.copy()
    shap_method = f'Fallback: {imp_method}'

print('Primary importance method ->', imp_method)
print('Second panel method ->', shap_method)
rf_imp


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.2))

axes[0].barh(rf_imp['feature'][::-1], rf_imp['importance'][::-1], color=PALETTE['myred'])
axes[0].set_title(f'Random Forest importance ({imp_method})')
axes[0].set_xlabel('Importance score')

axes[1].barh(rf_shap['feature'][::-1], rf_shap['importance'][::-1], color=PALETTE['myblue'])
axes[1].set_title(f'Random Forest importance ({shap_method})')
axes[1].set_xlabel('Score')

fig.tight_layout()
plt.show()


## 7. When could ML work better?

Machine learning tends to become more attractive when at least one of these is true:

1. **more data** — many countries, many more time periods, or a richer panel,
2. **more predictors** — text, financial variables, alternative data,
3. **nonlinear effects** — thresholds, interactions, regime changes,
4. **stronger tuning discipline** — cross-validation, feature selection, regularization.

In a small single-country quarterly sample, it is perfectly normal for ML to lose to a strong structured benchmark.

That is not a failure. It is a result.
